In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType, FloatType, LongType, BooleanType, DoubleType

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, IndexToString, SQLTransformer, Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics

from pyspark.ml.classification import LogisticRegression

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 15:54:58 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/05/25 15:54:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 15:54:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
inputData = "data/Reviews.csv"
outputPath = "output/"

In [3]:
reviews = spark.read.load(inputData,\
                     format="csv",\
                     header=True,\
                     inferSchema=True)

In [4]:
reviews.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- ProductId: string (nullable = true)
 |-- UserId: string (nullable = true)
 |-- ProfileName: string (nullable = true)
 |-- HelpfulnessNumerator: integer (nullable = true)
 |-- HelpfulnessDenominator: integer (nullable = true)
 |-- Score: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- Summary: string (nullable = true)
 |-- Text: string (nullable = true)



In [5]:
# Select only the records with HelpfulnessDenominator>0 (i.e., rated reviews)
rated_reviews = reviews.filter("HelpfulnessDenominator>0")

In [6]:
# Create and compute the value of Column label for the selected rated reviews
# For this task, a review belongs to the “useful” class if its helpfulness index is above 90% (0.9).
# Store the result of this step in a new Dataframe called reviewsWithLabel
def isHelpful(HelpfulnessNumerator, HelpfulnessDenominator):
    if HelpfulnessNumerator / HelpfulnessDenominator > 0.9:
        return 1
    else:
        return 0

spark.udf.register("isHelpful", isHelpful, IntegerType())

reviewsWithLabel = rated_reviews \
    .selectExpr(
        "*", \
        "isHelpful(HelpfulnessNumerator, HelpfulnessDenominator)" \
    ) \
    .withColumnRenamed("isHelpful(HelpfulnessNumerator, HelpfulnessDenominator)", "label") \
    .na.drop(subset=["Text", "Summary"])

In [7]:
reviewsWithLabel.show()

+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+-----+
| Id| ProductId|        UserId|         ProfileName|HelpfulnessNumerator|HelpfulnessDenominator|Score|      Time|             Summary|                Text|label|
+---+----------+--------------+--------------------+--------------------+----------------------+-----+----------+--------------------+--------------------+-----+
|  1|B001E4KFG0|A3SGXH7AUHU8GW|          delmartian|                   1|                     1|    5|1303862400|Good Quality Dog ...|I have bought sev...|    1|
|  3|B000LQOCH0| ABXLMWJIXXAIN|"Natalia Corres "...|                   1|                     1|    4|1219017600|"""Delight"" says...|"This is a confec...|    1|
|  4|B000UA0QIQ|A395BORC6FGVXV|                Karl|                   3|                     3|    2|1307923200|      Cough Medicine|If you are lookin...|    1|
|  9|B000E7L2R4|A1MZYO9TZK0B

In [8]:
# Split the dataframe with Column label in training and test set
(reviews_train, reviews_test) = reviewsWithLabel.randomSplit([0.75,0.25], seed=10)

In [12]:
# Create/Define the preprocessing steps and the classification algorithm you want to use 
# and the content of the pipeline that is used to train the model on reviews_train and apply it on reviews_test
# Implement a first solution with one single values in features: text length

def get_text_stages_features(col_name, num_features=1000):

    col_name = col_name.lower()
    
    tokenizer = Tokenizer() \
        .setInputCol(col_name) \
        .setOutputCol(f"raw_{col_name}_words")

    stopwords_remover = StopWordsRemover() \
        .setInputCol(f"raw_{col_name}_words") \
        .setOutputCol(f"filtered_{col_name}_words")

    hashing_tf = HashingTF() \
        .setInputCol(f"filtered_{col_name}_words") \
        .setOutputCol(f"{col_name}_tf") \
        .setNumFeatures(num_features)

    idf = IDF() \
        .setInputCol(f"{col_name}_tf") \
        .setOutputCol(f"{col_name}_tf_idf")

    len_sql_transformer = SQLTransformer(
        statement=f"SELECT *, len({col_name}) AS {col_name}_len FROM __THIS__"
    )

    return [tokenizer, stopwords_remover, hashing_tf, idf, len_sql_transformer], (f"{col_name}_len", f"{col_name}_tf_idf")

text_stages, text_features = get_text_stages_features("Text", num_features=1000)
summary_stages, summary_features = get_text_stages_features("Summary", num_features=500)

assembler = VectorAssembler() \
    .setInputCols([
        'Score', 
        *text_features,
        *summary_features,
    ]) \
    .setOutputCol("features")

lr = LogisticRegression(
    featuresCol='features',
    labelCol='label'
)
lr.setMaxIter(10)
lr.setRegParam(0.01)

pipeline = Pipeline().setStages([
    *text_stages,
    *summary_stages,
    assembler,
    lr
])

In [13]:
# Fit/Train the model
model = pipeline.fit(reviews_train)

26/05/25 15:55:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [14]:
# Apply the model on the test set
predictions = model.transform(reviews_test).cache()

In [15]:
# Compute statistics
# Accuracy, F1, weighted recall, weighted precision
evaluatorAcc = MulticlassClassificationEvaluator(labelCol="label" , predictionCol= "prediction", metricName = "accuracy")
evaluatorF1 = MulticlassClassificationEvaluator(labelCol="label" , predictionCol= "prediction", metricName = "f1")
evaluatorRecall = MulticlassClassificationEvaluator(labelCol="label" , predictionCol= "prediction", metricName = "weightedRecall")
evaluatorPrecision = MulticlassClassificationEvaluator(labelCol="label" , predictionCol= "prediction", metricName = "weightedPrecision")

print("Accuracy:", evaluatorAcc.evaluate(predictions))
print("F1:", evaluatorF1.evaluate(predictions))
print("Weighted Recall:", evaluatorRecall.evaluate(predictions))
print("Weighted Precision:", evaluatorPrecision.evaluate(predictions))

Accuracy: 0.7286227480936419
F1: 0.7136847585926711
Weighted Recall: 0.7286227480936419
Weighted Precision: 0.7236828825818925


In [16]:
#  Compute the confusion matrix
#                     Predicted  
#  Actual       Useful   Useless
#  Useful          A        B
#  Useless          C        D

A = predictions.filter("prediction=1 and label=1").count()
B = predictions.filter("prediction=0 and label=1").count()
C = predictions.filter("prediction=1 and label=0").count()
D = predictions.filter("prediction=0 and label=0").count()

print("                       Predicted")
print("  Actual \t Useful\tUseless")
print("  Useful \t "+str(A)+ "\t\t"+str(B))
print("  Useless \t "+str(C)+ "\t\t"+str(D))

                       Predicted
  Actual 	 Useful	Useless
  Useful 	 41648		5581
  Useless 	 14740		12912


In [17]:
# Precision and recall for the two classes
# Useful
if A+C==0:
    print("Precision(Useful): undefined")
else:
    print("Precision(Useful):"+str(A/(A+C)))
    
    
print("Recall(Useful):"+str(A/(A+B)))

# Useless 
if B+D==0:
    print("Precision(Useless): undefined")
else:
    print("Precision(Useless):"+str(D/(B+D)))
    
print("Recall(Useless):"+str(D/(C+D)))

Precision(Useful):0.7385968645811165
Recall(Useful):0.8818310783628702
Precision(Useless):0.6982101335640513
Recall(Useless):0.466946332995805
